In [0]:
# 4️⃣ Customer Semantics
# Load silver-layer customers and orders tables

df_customers = spark.table("retail_catalog.silver.customers")
df_orders = spark.table("retail_catalog.silver.orders")

In [0]:
from pyspark.sql.functions import col
# Filter out customers with null customer_id for integrity
df_customers = df_customers.filter((col("customer_id").isNotNull()))

In [0]:
# Join customers and orders on customer_id to build a combined customer-orders view
df_customers_orders = df_customers.join(df_orders, on="customer_id", how="inner")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Aggregate customer behavior from orders
customer_behavior = (
    df_customers_orders
    .groupBy("customer_unique_id")
    .agg(
        # F.first("customer_id").alias("customer_id"), # Not needed for unique_id aggregation
        F.countDistinct("order_id").alias("total_orders"), # Count unique orders per customer
        F.min("order_purchase_timestamp").alias("first_order_ts"), # Earliest order timestamp
        F.max("order_purchase_timestamp").alias("last_order_ts") # Latest order timestamp
    )
)

# Uncomment to display the aggregated behavior metrics
# customer_behavior.display()

In [0]:
# Derive customer type: 'NEW' if only one order, otherwise 'RETURNING'
customer_behavior = customer_behavior.withColumn(
    "customer_type",
    F.when(F.col("total_orders") == 1, "NEW")
     .otherwise("RETURNING")
)

# Join behavior metrics back to customer-orders view for semantic enrichment
df_final = (
    df_customers_orders
    .join(
        customer_behavior,
        on="customer_unique_id",
        how="left"
    )
)

# Uncomment to display final enriched DataFrame
# df_final.display()

In [0]:
# Select semantic columns for gold-layer customer model
customer_semantic = df_final.select(
    "customer_id",
    "customer_unique_id",
    "customer_state",
    "order_id",
    "order_status",
    "total_orders",
    "first_order_ts",
    "last_order_ts",
    "customer_type"
)

# Uncomment to display semantic DataFrame for review
# display(customer_semantic)

In [0]:
# Final semantic customer DataFrame; ready for further ETL, analytics, or persistence
customer_semantic

DataFrame[customer_id: string, customer_unique_id: string, customer_state: string, order_id: string, order_status: string, total_orders: bigint, first_order_ts: timestamp, last_order_ts: timestamp, customer_type: string]